# Equatorial Pacific

This notebook evaluates the equatorial Pacific Ocean circulation in ACCESS-OM3, comparing it to ACCESS-OM2 and to observations. It produces three figures:

1. **Temperature and zonal velocity structure**: depth-longitude sections along the equator and depth-latitude sections at 140°W, compared to the Johnson et al. (2002) climatology. This reproduces Fig 19 of the [GMD ACCESS-OM2 paper](https://gmd.copernicus.org/articles/13/401/2020/).
2. **Temperature bias along the equator**: a depth-longitude section of model-minus-observations temperature bias along the equator, compared to the [World Ocean Atlas 2023](https://www.ncei.noaa.gov/products/world-ocean-atlas) (WOA23) climatology.
3. **Temperature and zonal velocity profiles**: vertical profiles at three long-running equatorial moorings (165°E, 140°W, 110°W), compared to the [TAO/TRITON](https://www.pmel.noaa.gov/tao/drupal/disdel/) array (with Johnson et al. (2002) and WOA23 shown for additional context).

By default this notebook compares the RYF (repeat-year forcing) OM3 and OM2 runs, averaged over the last `averaging_last_n_years` years of each run (see `averaging_mode` below), following the README's guidance and the convention used in `SST.ipynb`/`SSS.ipynb`/`MLD.ipynb`. The corresponding IAF (interannually forced) experiments are also wired up and can be swapped in by editing `esm_file` and `om2_experiment`.

In [ ]:
# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in 
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest.
# RYF (repeat-year forcing) is the default here, following the convention in
# SST.ipynb/SSS.ipynb/MLD.ipynb and the README's "average over the last 10
# years of the RYF run" guidance -- RYF spin-ups are much shorter to process
# than the multi-decade IAF runs below. `averaging_mode`/`averaging_last_n_years`
# further down apply equally to either choice, so swap the comment below to
# compare against IAF instead.
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_ryf+wombatlite-test3-f4d79e82/experiment_datastore.json"
# esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test3v2-00532b88/datastore.json"  # IAF alternative

# papermill settings. *No need to modify these if running interactively.* 
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory 
nbname = None                          # notebook name

In [ ]:
import os
if not papermill: 
    import nci_ipynb  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)

In [ ]:
IAF = esm_file.find('iaf') > 0
IAF

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import cftime
import intake
import matplotlib.pyplot as plt
from distributed import Client
import cmocean as cm

In [ ]:
client = Client(threads_per_worker=1)
client

### Open the intake-esm datastore

In [ ]:
COLUMNS_WITH_ITERABLES = [
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
]

datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)

### What ocean variables are available at monthly frequency?

In [ ]:
def available_variables(datastore):
    """Return a pandas dataframe summarising the variables in a datastore"""
    variable_columns = [col for col in datastore.df.columns if "variable" in col]
    return (
        datastore.df[variable_columns]
        .explode(variable_columns)
        .drop_duplicates()
        .set_index("variable")
        .sort_index()
    )

In [ ]:
datastore_filtered = datastore.search(realm="ocean", frequency="1mon", variable="uo")

available_variables(datastore_filtered)

### ACCESS-OM2 dataset

In [ ]:
cat = intake.cat.access_nri

# RYF (repeat-year forcing), to match the OM3 RYF choice above and keep
# processing fast. Swap the comment below to compare against the IAF cycle
# used previously instead.
om2_experiment = "025deg_jra55_ryf9091_gadi"
# om2_experiment = "025deg_jra55_iaf_omip2_cycle6"  # IAF alternative

om2_datastore = cat[om2_experiment]

In [ ]:
temp_om2 = om2_datastore.search(variable="temp", frequency="1mon").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"yt_ocean": -1, "xt_ocean": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True,
        use_cftime=True,  # RYF spin-ups run out past numpy.datetime64[ns]'s ~2262 limit
    )
)

u_om2 = om2_datastore.search(variable="u", frequency="1mon").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"yu_ocean": -1, "xu_ocean": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True,
        use_cftime=True,  # RYF spin-ups run out past numpy.datetime64[ns]'s ~2262 limit
    )
)

### Rename OM2 var name and coords to be consistent with OM3 for plotting purpose

In [ ]:
temp_om2 = temp_om2.rename(
    {
        "st_ocean": "z_l",
        "yt_ocean": "yh",
        "xt_ocean": "xh",
        "temp": "thetao"
    }
)

u_om2 = u_om2.rename(
    {
        "st_ocean": "z_l",
        "yu_ocean": "yh",
        "xu_ocean": "xq",
        "u": "uo"
    }
)

### ACCESS-OM3 dataset

In [ ]:
temp_om3 = datastore.search(variable="thetao", frequency="1mon").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"yh": -1, "xh": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True,
        use_cftime=True,  # RYF spin-ups run out past numpy.datetime64[ns]'s ~2262 limit
    )
)

u_om3 = datastore.search(variable="uo", frequency="1mon").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"yh": -1, "xq": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True,
        use_cftime=True,  # RYF spin-ups run out past numpy.datetime64[ns]'s ~2262 limit
    )
)

#### NOTE: There's an [ordering issue](https://github.com/ACCESS-NRI/access-om3-configs/pull/725#pullrequestreview-3174863908) in `yh`. This should / will be fixed.

In [ ]:
temp_om3 = temp_om3.assign_coords(
    {"yh": temp_om3.yh.sortby("yh")}
)
u_om3 = u_om3.assign_coords(
    {"yh": u_om3.yh.sortby("yh")}
)

### Observation dataset

In [ ]:
obs_path = '/g/data/v45/fw4078/obs-data/meanfit_m.cdf'
obs_file = xr.open_dataset(obs_path)

In [ ]:
obs_file

In [ ]:
temp_obs = obs_file["POTEMPM"].to_dataset()
u_obs = obs_file["UM"].to_dataset()

In [ ]:
temp_obs = temp_obs.rename(
    {
        "ZDEP1_50": "z_l",
        "YLAT11_101": "yh",
        "XLON": "xh",
        "POTEMPM": "thetao"
    }
)

u_obs = u_obs.rename(
    {
        "ZDEP1_50": "z_l",
        "YLAT11_101": "yh",
        "XLON": "xq",
        "UM": "uo"
    }
)

In [ ]:
u_obs

#### Extract Equatorial Pacific region

In [ ]:
def _time_detection(da, tsel):
    """
    observation data does not contain time coord.
    """
    if "time" in da.dims:
        return da.sel(time=tsel).mean("time")
    else:
        return da

def make_sections(
    temp,
    u,
    zsel=slice(0,300),
    xsel=slice(-217, -95),
    ysel=slice(-8,10),
    tsel=slice(None, None),  # supply explicitly for real model data -- see averaging_window() below
    x_140w=-140.0,
    yeq=0,
    u_scale=100.0
):
    temp_eq = _time_detection(
        temp.sel(z_l=zsel,xh=xsel).sel(yh=0, method='nearest'), tsel,
    ).compute()
    u_eq = u_scale*_time_detection(
        u.sel(z_l=zsel,xq=xsel).sel(yh=0, method='nearest'), tsel,
    ).compute()
    temp_140w = _time_detection(
        temp.sel(z_l=zsel,yh=ysel).sel(xh=x_140w, method='nearest'), tsel,
    ).compute()
    u_140w = _time_detection(
        u_scale*u.sel(z_l=zsel,yh=ysel).sel(xq=x_140w, method='nearest'), tsel,
    ).compute()

    return temp_eq, u_eq, temp_140w, u_140w

In [ ]:
# How much of the model run to average over. Follows the same convention as
# SST.ipynb / SSS.ipynb / MLD.ipynb: "last_n_years" (the README's guidance,
# and much cheaper to process than the full IAF record) or "full_period"
# (average everything available -- e.g. to reproduce the old 1998-onward IAF
# behaviour more closely). Applies independently to whichever OM2/OM3
# experiment is configured above (RYF or IAF).
averaging_mode = "last_n_years"
averaging_last_n_years = 10

def averaging_window(time):
    """
    Return a (start, stop) tuple spanning the configured averaging window,
    computed relative to `time`'s own final timestep (not a fixed real-world
    date). This means it works unmodified whether `time` uses a real
    calendar (IAF) or the synthetic multi-century calendar typical of RYF
    spin-ups -- and it's why observational products (Johnson, WOA23, TAO)
    are *not* tied to this window below: there's no meaningful way to line
    up a RYF spin-up's synthetic dates with a real observational period.
    """
    t0, t1 = time.values[0], time.values[-1]
    if averaging_mode == "full_period":
        return t0, t1
    elif averaging_mode == "last_n_years":
        datelist = list(cftime.to_tuple(t1))
        datelist[0] -= averaging_last_n_years
        return cftime.datetime(*datelist, calendar=t1.calendar), t1
    else:
        raise ValueError(f"Unknown averaging_mode: {averaging_mode!r}")

#### ACCESS-OM2 temperature and zonal velocity

In [ ]:
temp_om2 = temp_om2.convert_calendar("proleptic_gregorian", use_cftime=True)
u_om2 = u_om2.convert_calendar("proleptic_gregorian", use_cftime=True)

tsel_om2 = slice(*averaging_window(temp_om2.time))
print("OM2 averaging window:", tsel_om2)

temp_eq_om2, u_eq_om2, temp_140w_om2, u_140w_om2 = make_sections(temp_om2, u_om2, tsel=tsel_om2)
temp_eq_om2 = temp_eq_om2 - 273.15
temp_140w_om2 = temp_140w_om2 - 273.15

#### ACCESS-OM3 temperature and zonal velocity

In [ ]:
temp_om3 = temp_om3.convert_calendar("proleptic_gregorian", use_cftime=True)
u_om3 = u_om3.convert_calendar("proleptic_gregorian", use_cftime=True)

tsel_om3 = slice(*averaging_window(temp_om3.time))
print("OM3 averaging window:", tsel_om3)

temp_eq_om3, u_eq_om3, temp_140w_om3, u_140w_om3 = make_sections(temp_om3, u_om3, tsel=tsel_om3)

#### Observation temperature and zonal velocity

In [ ]:
xsel1 = temp_obs.xh.values[0]
xsel2 = temp_obs.xh.values[-1]

temp_eq_obs, u_eq_obs, temp_140w_obs, u_140w_obs = make_sections(temp_obs, u_obs,
                                                                 xsel=slice(xsel1, xsel2),
                                                                 ysel=slice(-8, 10),
                                                                 x_140w=220
                                                                )
                                                                 

### Plot

In [ ]:
def plot_pair(ax,
              temp,
              u,
              xdim_temp, xdim_u,
              clevels_temp, clevels_u,
              title,
              show_ylabel=False,
              show_xlabel=False
             ):
    T = temp["thetao"]
    U = u["uo"]

    cf = T.plot.contourf(
        ax=ax,
        x=xdim_temp, y="z_l",
        levels=clevels_temp,
        cmap=cm.cm.thermal,
        extend="both",
        add_colorbar=False,
        zorder=1,
    )
    ct = T.plot.contour(
        ax=ax,
        x=xdim_temp, y="z_l",
        levels=clevels_temp,
        colors="k",
        linewidths=0.5,
        zorder=2,
    )
    cs = U.plot.contour(
        ax=ax,
        x=xdim_u, y="z_l",
        levels=clevels_u,
        colors="w",
        linewidths=1.0,
        zorder=3,
    )

    labels = ax.clabel(cs, fmt="%d", fontsize=9, inline=True)
    for t in labels:
        t.set_color("black")
        t.set_bbox(dict(facecolor="white", edgecolor="none", pad=0.6, alpha=0.75))

    ax.invert_yaxis()
    ax.set_title(title)

    if show_ylabel:
        ax.set_ylabel("Depth (m)")
    else:
        ax.set_ylabel("")

    if show_xlabel:
        ax.set_xlabel(xdim_temp)
    else:
        ax.set_xlabel("")

    ax.grid(False)
    return cf, ct, cs

In [ ]:
clevels_temp = np.arange(10., 30., 1.)
clevels_u = np.arange(-50, 130, 10)

fig, axs = plt.subplots(
    3, 2,
    figsize=(14, 9),
    sharey=True,
    constrained_layout=True
)

# OM3 row
cf00, ct00, _ = plot_pair(
    axs[0, 0],
    temp_eq_om3, u_eq_om3,
    xdim_temp="xh", xdim_u="xq",
    clevels_temp=clevels_temp,
    clevels_u=clevels_u,
    title="OM3 – Equator (y=0)",
    show_ylabel=True
)
plot_pair(
    axs[0, 1],
    temp_140w_om3, u_140w_om3,
    xdim_temp="yh", xdim_u="yh",
    clevels_temp=clevels_temp,
    clevels_u=clevels_u,
    title="OM3 – 140W (x=-140)"
)

# OM2 row
plot_pair(
    axs[1, 0],
    temp_eq_om2, u_eq_om2,
    xdim_temp="xh", xdim_u="xq",
    clevels_temp=clevels_temp,
    clevels_u=clevels_u,
    title="OM2 – Equator (y=0)",
    show_ylabel=True,
    # show_xlabel=True
)
plot_pair(
    axs[1, 1],
    temp_140w_om2, u_140w_om2,
    clevels_temp=clevels_temp,
    clevels_u=clevels_u,
    xdim_temp="yh", xdim_u="yh",
    title="OM2 – 140W (x=-140)",
)

# obs row
plot_pair(
    axs[2, 0],
    temp_eq_obs, u_eq_obs,
    xdim_temp="xh", xdim_u="xq",
    clevels_temp=clevels_temp,
    clevels_u=clevels_u,
    title="OBS – Equator (y=0)",
    show_ylabel=True,
    show_xlabel=True
)
plot_pair(
    axs[2, 1],
    temp_140w_obs, u_140w_obs,
    clevels_temp=clevels_temp,
    clevels_u=clevels_u,
    xdim_temp="yh", xdim_u="yh",
    title="OBS – 140W (x=-140)",
    show_xlabel=True
)

cbar = fig.colorbar(cf00, ax=axs, orientation="vertical", shrink=0.95, pad=0.02)
cbar.set_label("Temperature (°C)")



plt.show()
mkmd.savefig(fig, "Equatorial Pacific", "Contours of temperature and zonal velocity in the equatorial Pacific compared to observations from Johnson et al. (2002). [GitHub issue: Equatorial T & zonal velocity contours](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/17)")

## Temperature bias along the equator compared to WOA23

The panel above shows the raw thermal structure, but model-observation differences are easier to see directly as a bias. Here we plot the model-minus-observations temperature bias along the equator (depth vs longitude), using the [World Ocean Atlas 2023](https://www.ncei.noaa.gov/products/world-ocean-atlas) (WOA23) 2005-2014 decadal-average climatology as the reference, following the equatorial temperature bias plots used in earlier COSIMA/ACCESS-OM2 analyses.

**Note:** unlike the WOA13 data used in those earlier analyses (which was pre-interpolated onto each ACCESS-OM2 grid resolution, see `/g/data/hh5/tmp/cosima/woa13/`, now decommissioned), there is currently no version of WOA pre-interpolated onto the ACCESS-OM3 grid. Below, WOA23 is instead interpolated directly from its native 0.25° lat-lon grid onto each model's *equatorial transect* (matching longitude and depth) using `xarray.interp`. Because this only needs to produce a 1-D transect (not the full 3-D field), a lightweight point-wise interpolation is used rather than a full conservative regrid (e.g. via `xesmf`) — the same approach the existing Johnson et al. comparison above already relies on. This should be adequate for a line/section plot, but flag if a proper conservative regrid is preferred instead.

In [ ]:
# WOA23 temperature climatology (2005-2014 decadal average, "A5B4"), on its
# native 0.25 degree lat-lon grid with 102 standard depth levels.
woa_path = "/g/data/ik11/observations/woa23/woa23_A5B4_t00_04.nc"
woa_ds = xr.open_dataset(woa_path, decode_times=False)

# Restrict to the upper ocean near the equator before loading into memory.
temp_woa = woa_ds["t_an"].isel(time=0).sel(depth=slice(0, 400)).sel(lat=slice(-15, 15))

def to_0_360(lon):
    """Convert longitude(s) to the 0-360 convention used by WOA and the Johnson et al. dataset."""
    return lon % 360

# Re-express WOA's longitude on 0-360 (rather than -180-180) so that the
# Pacific equatorial transect, which crosses the dateline, is contiguous.
temp_woa = temp_woa.assign_coords(lon=to_0_360(temp_woa.lon)).sortby("lon")
temp_woa = temp_woa.load()

In [ ]:
def interp_woa_to_model(temp_woa, xh=None, lon=None, z_l=None, yh=0.0):
    """
    Interpolate the (0-360 longitude) WOA23 temperature climatology onto
    model-like coordinates for direct comparison.

    Pass `xh` (a model xh DataArray) for a longitude-depth transect, or a
    scalar `lon` (degrees East, 0-360 convention) for a single-column
    profile. Pass `z_l` (a model depth DataArray) to also interpolate onto
    specific model depth levels, e.g. to allow a direct model-minus-WOA
    subtraction; omit it to keep WOA's native depth levels (used for the
    profile plots below).
    """
    interp_kwargs = {"lat": yh}
    if xh is not None:
        interp_kwargs["lon"] = xr.DataArray(to_0_360(xh.values), dims="xh", coords={"xh": xh.values})
    else:
        interp_kwargs["lon"] = to_0_360(lon)
    if z_l is not None:
        interp_kwargs["depth"] = z_l
    return temp_woa.interp(**interp_kwargs)


temp_woa_eq_om3 = interp_woa_to_model(temp_woa, xh=temp_eq_om3.xh, z_l=temp_eq_om3.z_l)
temp_woa_eq_om2 = interp_woa_to_model(temp_woa, xh=temp_eq_om2.xh, z_l=temp_eq_om2.z_l)

bias_eq_om3 = temp_eq_om3["thetao"] - temp_woa_eq_om3
bias_eq_om2 = temp_eq_om2["thetao"] - temp_woa_eq_om2

In [ ]:
def plot_bias(ax, bias, woa_temp, clevels_bias, clevels_iso, title, show_ylabel=False, show_xlabel=False):
    cf = bias.plot.contourf(
        ax=ax, x="xh", y="z_l",
        levels=clevels_bias, cmap="RdBu_r", extend="both",
        add_colorbar=False, zorder=1,
    )
    ci = woa_temp.plot.contour(
        ax=ax, x="xh", y="z_l",
        levels=clevels_iso, colors="k", linewidths=0.5, zorder=2,
    )
    ax.clabel(ci, fmt="%d", fontsize=8, inline=True)
    # 20C isotherm from WOA23 (solid) and from the model (dashed), for a quick
    # visual check of thermocline depth/tilt biases.
    woa_temp.plot.contour(ax=ax, x="xh", y="z_l", levels=[20.], colors="k", linewidths=2.0, zorder=3)
    (bias + woa_temp).plot.contour(ax=ax, x="xh", y="z_l", levels=[20.], colors="k", linewidths=2.0, linestyles="--", zorder=3)

    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_ylabel("Depth (m)" if show_ylabel else "")
    ax.set_xlabel("Longitude" if show_xlabel else "")
    ax.grid(False)
    return cf


clevels_bias = np.arange(-3., 3.25, 0.25)
clevels_iso = np.arange(10., 32., 2.)

fig, axs = plt.subplots(2, 1, figsize=(9, 7), sharex=True, sharey=True, constrained_layout=True)

cf0 = plot_bias(
    axs[0], bias_eq_om3, temp_woa_eq_om3, clevels_bias, clevels_iso,
    "OM3 minus WOA23 – Equator (y=0)", show_ylabel=True,
)
plot_bias(
    axs[1], bias_eq_om2, temp_woa_eq_om2, clevels_bias, clevels_iso,
    "OM2 minus WOA23 – Equator (y=0)", show_ylabel=True, show_xlabel=True,
)

cbar = fig.colorbar(cf0, ax=axs, orientation="vertical", shrink=0.95, pad=0.02)
cbar.set_label("Temperature bias (°C)")

plt.show()
mkmd.savefig(
    fig,
    "Equatorial Pacific temperature bias",
    "Model-minus-WOA23 temperature bias (colour) along the equator, with WOA23 isotherms "
    "(thin black contours, every 2°C) and the 20°C isotherm from WOA23 (solid) and the model "
    "(dashed) overlaid. NOTE: no existing GitHub issue currently tracks this figure — please "
    "create one following the repo convention (one issue per figure) and update this caption/link."
)

## Temperature and zonal velocity profiles compared to the TAO array

A bias averaged along a section can also mask compensating errors at different depths. Here we plot vertical profiles of temperature and zonal velocity at three long-running TAO/TRITON mooring locations along the equator — 165°E, 140°W and 110°W — comparing ACCESS-OM3 and ACCESS-OM2 to the [TAO array](https://www.pmel.noaa.gov/tao/drupal/disdel/) daily mooring data (temperature from the mooring thermistors, zonal velocity from the moored ADCP). The Johnson et al. (2002) climatology and WOA23 (temperature only — WOA does not include velocity) are also shown for additional context. Note that each observational product averages over its own period (TAO's full daily record; Johnson's older shipboard/mooring composite; WOA23's 2005-2014 decadal mean) rather than being matched to the model's `averaging_window` — there's no meaningful way to line up a RYF spin-up's synthetic calendar with a real observational period, so the obs products are left as independent references rather than forced onto the model's window.

The OM3/OM2 model profiles below are sliced directly out of the equatorial sections already computed for the first figure (`temp_eq_om3`/`u_eq_om3`/`temp_eq_om2`/`u_eq_om2`) rather than re-querying the datastore, so this section adds no further model I/O — the trade-off is that the profiles are limited to the same 0-300 m depth range as that section, rather than the 400 m used in the original COSIMA notebook this analysis is based on.

TAO data are read from the daily-mean `.cdf` files in `/g/data/ik11/observations/TAO`. Following the approach in the original COSIMA equatorial Pacific notebook this analysis is based on, depths with too few valid daily observations over the averaging period are dropped from the time mean, since some instruments have patchy records.

In [ ]:
# Longitude of each mooring, expressed in the different conventions used by
# each dataset (model xh, Johnson/WOA lon [0-360 convention], TAO station code).
profile_stations = {
    "165E": {"om": -195.0, "obs_lon": 165.0, "tao": "165e"},
    "140W": {"om": -140.0, "obs_lon": 220.0, "tao": "140w"},
    "110W": {"om": -110.0, "obs_lon": 250.0, "tao": "110w"},
}

# Matches make_sections' default zsel (0-300 m), so the model profiles below
# can be sliced directly out of temp_eq_om3/u_eq_om3/temp_eq_om2/u_eq_om2
# (already computed for the first figure) instead of re-querying the lazy
# source datasets -- avoids re-reading the model output from disk a second
# time just for these three columns.
profile_zsel = slice(0, 300)


def make_profile(temp_eq, u_eq, lon):
    """
    Select a single-column temperature and zonal-velocity profile at the
    given longitude directly out of an already-computed equatorial section
    (`temp_eq_om3`/`u_eq_om3` or `temp_eq_om2`/`u_eq_om2` from `make_sections`
    above, which already cover the full xh range these mooring longitudes
    fall within). This is just in-memory indexing -- no dask compute, no
    disk I/O -- and the units/time-averaging/degC conversion are already
    handled upstream, so nothing further is needed here.
    """
    temp_prof = temp_eq["thetao"].sel(xh=lon, method="nearest")
    u_prof = u_eq["uo"].sel(xq=lon, method="nearest")
    return temp_prof, u_prof


def make_obs_profile(lon, zsel=profile_zsel):
    """Extract a Johnson et al. (2002) profile at the given longitude (0-360 convention, e.g. 220 for 140W)."""
    temp_prof = temp_obs["thetao"].sel(z_l=zsel).sel(yh=0, method="nearest").sel(xh=lon, method="nearest")
    # NOTE: the file's "UM" units attribute claims cm/s, but that's wrong --
    # checking the raw values against TAO ADCP at the same location confirms
    # they're actually in m/s (e.g. ~1.05 here vs TAO's ~105 cm/s at 140W,
    # 110m). So we apply the same *100 scaling as the model here, same as
    # `make_sections` already does for the first figure above.
    u_prof = 100.0 * u_obs["uo"].sel(z_l=zsel).sel(yh=0, method="nearest").sel(xq=lon, method="nearest")
    return temp_prof, u_prof


TAO_MIN_OBS = 2500  # min. number of valid daily obs at a depth to trust the time mean (following the original COSIMA notebook)

def make_tao_profile(station, zsel=profile_zsel, tsel=slice(None, None)):
    """
    Load daily TAO/TRITON temperature and ADCP zonal-velocity data at the
    equator for the given station code (e.g. '165e', '140w', '110w') and
    return the `tsel`-averaged profile. Depths with fewer than TAO_MIN_OBS
    valid daily observations over `tsel` are dropped before averaging.

    `tsel` defaults to TAO's full available record (like Johnson and WOA23,
    it is *not* matched to the model's `averaging_window` -- there's no
    meaningful way to line up a RYF spin-up's synthetic dates with a real
    observational period, and even for IAF there's no obvious "right" match
    once the model's own window is dynamic. TAO simply reports whatever
    period its own record actually covers, like the other obs products.)
    """
    temp_file = xr.open_dataset(f"/g/data/ik11/observations/TAO/t0n{station}_dy.cdf")
    temp = temp_file["T_20"].isel(lon=0, lat=0).sel(depth=zsel).sel(time=tsel)
    temp = temp.where(temp < 1.0e34)  # mask the ~1e35 fill value
    temp = temp.where(temp.notnull().sum("time") >= TAO_MIN_OBS, drop=True)
    temp_prof = temp.mean("time")

    u_file = xr.open_dataset(f"/g/data/ik11/observations/TAO/adcp0n{station}_dy.cdf")
    u = u_file["u_1205"].isel(lon=0, lat=0).sel(depth=zsel).sel(time=tsel)  # already cm/s
    u = u.where(np.abs(u) < 1.0e34)  # mask the ~1e35 fill value
    u = u.where(u.notnull().sum("time") >= TAO_MIN_OBS, drop=True)
    u_prof = u.mean("time")

    return temp_prof, u_prof

In [ ]:
profiles = {}
for label, loc in profile_stations.items():
    temp_om3_prof, u_om3_prof = make_profile(temp_eq_om3, u_eq_om3, loc["om"])
    temp_om2_prof, u_om2_prof = make_profile(temp_eq_om2, u_eq_om2, loc["om"])

    temp_johnson_prof, u_johnson_prof = make_obs_profile(loc["obs_lon"])
    temp_tao_prof, u_tao_prof = make_tao_profile(loc["tao"])
    temp_woa_prof = interp_woa_to_model(temp_woa, lon=loc["obs_lon"])

    profiles[label] = dict(
        om3=(temp_om3_prof, u_om3_prof),
        om2=(temp_om2_prof, u_om2_prof),
        johnson=(temp_johnson_prof, u_johnson_prof),
        tao=(temp_tao_prof, u_tao_prof),
        woa=temp_woa_prof,
    )

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(15, 12), sharey="row", constrained_layout=True)

for j, (label, p) in enumerate(profiles.items()):
    tax, uax = axs[0, j], axs[1, j]

    tax.plot(p["om3"][0], p["om3"][0].z_l, label="OM3", linewidth=2)
    tax.plot(p["om2"][0], p["om2"][0].z_l, label="OM2", linewidth=2)
    tax.plot(p["tao"][0], p["tao"][0].depth, "--", label="TAO", linewidth=2)
    tax.plot(p["johnson"][0], p["johnson"][0].z_l, "--", label="Johnson et al.", linewidth=2)
    tax.plot(p["woa"], p["woa"].depth, "--", label="WOA23", linewidth=2)
    tax.set_title(f"{label}, Temperature")
    tax.set_xlabel("Temperature (°C)")
    tax.set_ylabel("Depth (m)" if j == 0 else "")
    tax.set_ylim(300, 0)
    if j == 0:
        tax.legend(loc="lower right", fontsize=9)

    uax.plot(p["om3"][1], p["om3"][1].z_l, linewidth=2)
    uax.plot(p["om2"][1], p["om2"][1].z_l, linewidth=2)
    uax.plot(p["tao"][1], p["tao"][1].depth, "--", linewidth=2)
    uax.plot(p["johnson"][1], p["johnson"][1].z_l, "--", linewidth=2)
    uax.axvline(0, color="grey", linewidth=0.5)
    uax.set_title(f"{label}, Zonal velocity")
    uax.set_xlabel("Zonal velocity (cm s$^{-1}$)")
    uax.set_ylabel("Depth (m)" if j == 0 else "")
    uax.set_ylim(300, 0)

plt.show()
mkmd.savefig(
    fig,
    "Equatorial Pacific profiles",
    "Vertical profiles of temperature (top) and zonal velocity (bottom) at 165°E, 140°W and "
    "110°W, compared to the TAO array, Johnson et al. (2002) and (temperature only) WOA23. "
    "NOTE: no existing GitHub issue currently tracks this figure — please create one following "
    "the repo convention (one issue per figure) and update this caption/link."
)

In [ ]:
client.close()